# Notebook 4: Training and Fine-Tuning LLMs

## What You'll Learn
- How LLMs are pre-trained at scale (data, objectives, optimization)
- The three stages of creating a chatbot: pre-training → SFT → RLHF
- Fine-tune a real pre-trained model using HuggingFace Transformers
- LoRA: how to fine-tune billion-parameter models on consumer hardware
- Implement LoRA from scratch, then use PEFT for real fine-tuning

---

## The Three Stages of Creating a Chatbot

| Stage | What Happens | Data | Result |
|-------|-------------|------|--------|
| **1. Pre-training** | Train on massive text corpus | Internet-scale text (~1-15T tokens) | Base model (text completer) |
| **2. Supervised Fine-Tuning (SFT)** | Train on instruction-response pairs | ~10K-100K curated examples | Instruction-following model |
| **3. RLHF / DPO** | Align with human preferences | Human ranking data | Safe, helpful chatbot |

This notebook covers all three stages.

In [ ]:
# Install dependencies
# !pip install torch transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Pre-Training: How LLMs Learn Language

### The Objective

Pre-training uses a simple objective: **predict the next token**.

$$\mathcal{L} = -\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)$$

That's it. This deceptively simple objective, when applied to trillions of tokens, produces models that can reason, code, translate, and more.

### The Data

| Model | Training Data | Tokens |
|-------|--------------|--------|
| GPT-2 | WebText (Reddit links) | ~10B |
| GPT-3 | Common Crawl, Books, Wikipedia | 300B |
| LLaMA-2 | Public data + partnerships | 2T |
| LLaMA-3 | Broader web data | 15T |

### Key Training Techniques

1. **AdamW optimizer** with weight decay
2. **Learning rate warmup** + cosine decay
3. **Gradient accumulation** (simulate larger batch sizes)
4. **Mixed precision** (float16/bfloat16 for speed)
5. **Data parallelism** across many GPUs

In [ ]:
# Let's implement the key training techniques used by real LLMs

# 1. Learning Rate Schedule: Warmup + Cosine Decay
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    """Cosine learning rate schedule with warmup."""
    # Warmup phase: linear increase
    if step < warmup_steps:
        return max_lr * (step / warmup_steps)
    
    # After max steps
    if step >= max_steps:
        return min_lr
    
    # Cosine decay phase
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

# Visualize the schedule
max_steps = 10000
warmup_steps = 500
max_lr = 3e-4
min_lr = 3e-5

steps = range(max_steps)
lrs = [get_lr(s, warmup_steps, max_steps, max_lr, min_lr) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lrs)
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Cosine LR Schedule with Warmup (used by GPT-3, LLaMA, etc.)')
plt.axvline(x=warmup_steps, color='r', linestyle='--', label=f'Warmup ends ({warmup_steps} steps)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 2. Gradient Accumulation
# When you can't fit large batches in GPU memory, accumulate gradients over multiple mini-batches

def train_with_gradient_accumulation(model, data, batch_size, accum_steps, 
                                     num_steps, block_size, max_lr=3e-4):
    """
    Train with gradient accumulation.
    Effective batch size = batch_size * accum_steps
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.1)
    losses = []
    
    for step in range(num_steps):
        # Update learning rate
        lr = get_lr(step, warmup_steps=100, max_steps=num_steps, 
                    max_lr=max_lr, min_lr=max_lr/10)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        # Accumulate gradients over multiple mini-batches
        optimizer.zero_grad()
        total_loss = 0
        
        for micro_step in range(accum_steps):
            # Get mini-batch
            ix = torch.randint(len(data) - block_size, (batch_size,))
            xb = torch.stack([data[i:i+block_size] for i in ix]).to(device)
            yb = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
            
            # Forward pass
            _, loss = model(xb, targets=yb)
            
            # Scale loss by accumulation steps
            loss = loss / accum_steps
            loss.backward()
            total_loss += loss.item()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Update weights
        optimizer.step()
        
        losses.append(total_loss)
        
        if (step + 1) % 200 == 0:
            print(f'Step {step+1:5d} | Loss: {total_loss:.4f} | LR: {lr:.2e}')
    
    return losses

print("Gradient accumulation: simulate batch_size=64 with 4 steps of batch_size=16")
print("This makes large-batch training possible on limited GPU memory.")

## 2. Supervised Fine-Tuning (SFT)

After pre-training, the model is a powerful **text completer** but doesn't follow instructions.

**SFT** teaches the model to follow a specific format:

```
### Instruction: What is the capital of France?
### Response: The capital of France is Paris.
```

### How SFT Works

1. Collect instruction-response pairs (human-written or generated)
2. Format them in a template (chat format)
3. Fine-tune the model on this data, computing loss **only on the response tokens**

Let's implement this from scratch first, then use HuggingFace.

In [ ]:
# Demonstrate the concept of SFT with loss masking

# The key insight: during SFT, we ONLY compute loss on response tokens
# We don't want the model to memorize the prompts

def create_sft_training_example(instruction, response, tokenizer):
    """
    Create a training example with masked loss.
    Loss is only computed on response tokens.
    """
    prompt = f"### Instruction: {instruction}\n### Response: "
    full_text = prompt + response
    
    # Tokenize
    prompt_tokens = tokenizer(prompt)
    full_tokens = tokenizer(full_text)
    
    # Create labels: -100 for prompt tokens (ignored by cross-entropy)
    labels = [-100] * len(prompt_tokens) + full_tokens[len(prompt_tokens):]
    
    return full_tokens, labels

# Simple character-level tokenizer for demonstration
demo_text = "### Instruction: What is 2+2?\n### Response: 4"
prompt_part = "### Instruction: What is 2+2?\n### Response: "
response_part = "4"

print(f'Full text: "{demo_text}"')
print(f'\nPrompt length:   {len(prompt_part)} chars')
print(f'Response length: {len(response_part)} chars')
print(f'\nDuring SFT, loss is computed ONLY on the response ("4")')
print(f'The prompt tokens have their labels set to -100 (ignored)')
print(f'\nThis prevents the model from just memorizing prompts.')

In [ ]:
# Visualize the loss masking

prompt = "### Instruction: Explain gravity\n### Response: "
response = "Gravity is a force that pulls objects toward each other."

fig, ax = plt.subplots(figsize=(16, 3))

full_text = prompt + response
colors = ['lightcoral' if i < len(prompt) else 'lightgreen' for i in range(len(full_text))]

# Show each character as a colored block
for i, (char, color) in enumerate(zip(full_text, colors)):
    ax.barh(0, 1, left=i, color=color, edgecolor='white', linewidth=0.5)

ax.set_xlim(0, len(full_text))
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel('Token Position')

# Legend
import matplotlib.patches as mpatches
prompt_patch = mpatches.Patch(color='lightcoral', label='Prompt (loss masked, label=-100)')
response_patch = mpatches.Patch(color='lightgreen', label='Response (loss computed here)')
ax.legend(handles=[prompt_patch, response_patch], loc='upper right')

ax.set_title('SFT Loss Masking: Only Response Tokens Contribute to Loss')
plt.tight_layout()
plt.show()

## 3. LoRA: Efficient Fine-Tuning

Full fine-tuning updates **all** model parameters, which is:
- Memory intensive (need to store gradients for all params)
- Expensive (takes many GPU-hours)
- Risky (can destroy pre-trained knowledge)

### LoRA (Low-Rank Adaptation)

Key insight from [Hu et al., 2021]: weight updates during fine-tuning have **low intrinsic rank**. Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times d}$, we decompose the update:

$$W' = W + \Delta W = W + BA$$

Where:
- $B \in \mathbb{R}^{d \times r}$ (down-projection)
- $A \in \mathbb{R}^{r \times d}$ (up-projection)
- $r \ll d$ (rank, typically 4-64)

Parameters:
- Full fine-tuning: $d \times d$ parameters
- LoRA: $d \times r + r \times d = 2dr$ parameters
- With $d = 4096, r = 16$: **full = 16.7M**, **LoRA = 131K** (0.8% of full!)

In [ ]:
# Implement LoRA from scratch

class LoRALinear(nn.Module):
    """A linear layer with LoRA adaptation."""
    
    def __init__(self, original_linear, rank=4, alpha=1.0):
        super().__init__()
        
        self.original = original_linear
        in_features = original_linear.in_features
        out_features = original_linear.out_features
        
        # Freeze original weights
        for param in self.original.parameters():
            param.requires_grad = False
        
        # LoRA matrices
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))  # Initialize B to zero!
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def forward(self, x):
        # Original output (frozen)
        original_out = self.original(x)
        
        # LoRA output: x @ A @ B * scaling
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scaling
        
        return original_out + lora_out

# Demonstrate LoRA efficiency
d = 4096  # Typical LLM hidden dimension
rank = 16

original = nn.Linear(d, d)
lora_layer = LoRALinear(original, rank=rank)

original_params = sum(p.numel() for p in original.parameters())
lora_trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
lora_total = sum(p.numel() for p in lora_layer.parameters())

print(f'Original layer parameters: {original_params:,}')
print(f'LoRA trainable parameters: {lora_trainable:,}')
print(f'Reduction: {original_params / lora_trainable:.1f}x fewer trainable params')
print(f'LoRA is training only {lora_trainable / original_params * 100:.2f}% of parameters')

In [ ]:
# Demonstrate that LoRA output starts identical to original
# (because B is initialized to zeros)

x = torch.randn(2, d)

with torch.no_grad():
    original_out = original(x)
    lora_out = lora_layer(x)

diff = (original_out - lora_out).abs().max().item()
print(f'Max difference between original and LoRA output: {diff:.10f}')
print(f'(Should be ~0 because B is initialized to zeros)')
print(f'\nThis means: at the start of fine-tuning, the model behaves EXACTLY like the original.')
print(f'The LoRA matrices gradually learn the task-specific adaptation.')

In [ ]:
# Apply LoRA to a full model and compare parameter counts

def apply_lora_to_model(model, rank=4, alpha=1.0, target_modules=None):
    """
    Replace linear layers in the model with LoRA-augmented versions.
    Only the LoRA parameters will be trainable.
    """
    if target_modules is None:
        target_modules = ['qkv_proj', 'out_proj']  # Attention layers
    
    lora_count = 0
    for name, module in model.named_modules():
        for attr_name in list(dir(module)):
            if attr_name in target_modules:
                original = getattr(module, attr_name)
                if isinstance(original, nn.Linear):
                    lora = LoRALinear(original, rank=rank, alpha=alpha)
                    setattr(module, attr_name, lora)
                    lora_count += 1
    
    return lora_count

# Visualize LoRA parameter savings for different model sizes
model_sizes = {
    'GPT-2 Small (124M)': 124_000_000,
    'LLaMA-7B': 7_000_000_000,
    'LLaMA-13B': 13_000_000_000,
    'LLaMA-70B': 70_000_000_000,
}

ranks = [4, 8, 16, 32, 64]

print(f'{"Model":<20} | {"Full Params":<15} | {"LoRA r=16":<15} | {"% Trainable":<12}')
print('-' * 70)

for name, total in model_sizes.items():
    # Rough estimate: attention params are ~1/3 of total, LoRA on Q,V = ~1/6
    # With rank 16, each LoRA pair adds 2 * d * r params
    lora_params = total * 0.01  # ~1% is typical for LoRA
    pct = lora_params / total * 100
    print(f'{name:<20} | {total:>13,} | {int(lora_params):>13,} | {pct:>10.2f}%')

## 4. Fine-Tuning with HuggingFace — Real Implementation

Let's fine-tune a real pre-trained model. We'll use a small model (GPT-2) to demonstrate the process.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load a pre-trained model
model_name = "gpt2"  # 124M parameters — small enough for a laptop

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {model_name}')
print(f'Total parameters: {total_params:,}')
print(f'Vocabulary size: {tokenizer.vocab_size:,}')

In [ ]:
# Test the base model — it's a text completer, NOT a chatbot

def generate_text(model, tokenizer, prompt, max_new_tokens=100, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try asking it a question — it won't answer properly
prompts = [
    "Question: What is the capital of France?\nAnswer:",
    "### Instruction: Explain what machine learning is.\n### Response:",
    "The meaning of life is",
]

for prompt in prompts:
    print(f'\n{"="*60}')
    print(f'Prompt: {prompt}')
    print(f'{"="*60}')
    output = generate_text(model, tokenizer, prompt, max_new_tokens=80)
    print(output)

In [ ]:
# Prepare instruction tuning data

sft_data = [
    {"instruction": "What is the capital of France?", 
     "response": "The capital of France is Paris."},
    {"instruction": "Explain what machine learning is in simple terms.",
     "response": "Machine learning is a type of artificial intelligence where computers learn patterns from data instead of being explicitly programmed. It's like teaching by example rather than giving step-by-step instructions."},
    {"instruction": "Write a haiku about programming.",
     "response": "Code flows like water\nBugs hide in the deepest lines\nDebug, compile, run"},
    {"instruction": "What is 15 multiplied by 7?",
     "response": "15 multiplied by 7 is 105."},
    {"instruction": "List three benefits of exercise.",
     "response": "Three benefits of exercise are: 1) Improved cardiovascular health, 2) Better mental health and reduced stress, 3) Increased strength and flexibility."},
    {"instruction": "Translate 'hello' to Spanish.",
     "response": "'Hello' in Spanish is 'hola'."},
    {"instruction": "What causes rain?",
     "response": "Rain is caused by water evaporating from the Earth's surface, rising into the atmosphere as water vapor, cooling and condensing into tiny droplets that form clouds. When the droplets become heavy enough, they fall as rain."},
    {"instruction": "Write Python code to reverse a string.",
     "response": "Here's how to reverse a string in Python:\n\ndef reverse_string(s):\n    return s[::-1]\n\n# Example\nprint(reverse_string('hello'))  # Output: 'olleh'"},
    {"instruction": "Who wrote Romeo and Juliet?",
     "response": "Romeo and Juliet was written by William Shakespeare, believed to have been written between 1591 and 1596."},
    {"instruction": "Explain the difference between a list and a tuple in Python.",
     "response": "The main differences are: Lists are mutable (can be changed after creation) and use square brackets []. Tuples are immutable (cannot be changed) and use parentheses (). Tuples are slightly faster and use less memory."},
]

# Format into training strings
def format_sft_example(example):
    return f"### Instruction: {example['instruction']}\n### Response: {example['response']}{tokenizer.eos_token}"

formatted_examples = [format_sft_example(ex) for ex in sft_data]

print(f'Number of training examples: {len(formatted_examples)}')
print(f'\nExample formatted input:')
print(formatted_examples[0])

In [ ]:
# Apply LoRA using the PEFT library
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # Rank
    lora_alpha=32,                # Scaling factor
    lora_dropout=0.05,            # Dropout on LoRA layers
    target_modules=["c_attn"],    # Which layers to apply LoRA to (GPT-2 attention)
)

# Apply LoRA to the model
peft_model = get_peft_model(model, lora_config)

# Show parameter counts
peft_model.print_trainable_parameters()

In [ ]:
# Fine-tune with LoRA

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=2e-4, weight_decay=0.01)
peft_model.train()

losses = []
num_epochs = 20

for epoch in range(num_epochs):
    epoch_loss = 0
    
    for example in formatted_examples:
        # Tokenize
        inputs = tokenizer(example, return_tensors="pt", padding=True, 
                          truncation=True, max_length=256)
        inputs = {k: v.to(peft_model.device) for k, v in inputs.items()}
        
        # Forward pass (labels = input_ids for causal LM)
        outputs = peft_model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(formatted_examples)
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SFT Fine-Tuning Loss (with LoRA)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Test the fine-tuned model

peft_model.eval()

test_prompts = [
    "### Instruction: What is the capital of France?\n### Response:",
    "### Instruction: Explain what machine learning is in simple terms.\n### Response:",
    "### Instruction: What is 23 plus 45?\n### Response:",
    "### Instruction: Write a short poem about coding.\n### Response:",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)
    
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'\n{"="*60}')
    print(result)

print(f'\n\nNote: With only {len(sft_data)} training examples, the model is limited.')
print(f'Real SFT uses 10K-100K high-quality examples.')

## 5. RLHF and DPO: Aligning with Human Preferences

After SFT, the model follows instructions but might give:
- Harmful or dangerous responses
- Verbose or unhelpful answers  
- Confidently wrong information

### RLHF (Reinforcement Learning from Human Feedback)

1. **Collect comparison data**: Show humans multiple model responses, have them rank best to worst
2. **Train a reward model**: A neural network that predicts human preference scores
3. **Optimize with PPO**: Use the reward model to guide the LLM's training

### DPO (Direct Preference Optimization)

A simpler alternative to RLHF that skips the reward model:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

Where $y_w$ = preferred response, $y_l$ = rejected response.

In [ ]:
# Implement a simple reward model concept

class SimpleRewardModel(nn.Module):
    """A reward model that scores how 'good' a response is."""
    
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1)  # Single scalar reward
        )
    
    def forward(self, hidden_states):
        # Use the last token's hidden state as the sequence representation
        last_hidden = hidden_states[:, -1, :]  # (batch, d_model)
        reward = self.net(last_hidden)  # (batch, 1)
        return reward.squeeze(-1)  # (batch,)

# Demonstrate preference learning
print("=== RLHF Preference Data Example ===")
print()
print("Prompt: 'How do I make a cake?'")
print()
print("Response A (PREFERRED):")
print("  'To make a basic cake, you'll need flour, sugar, eggs, and butter.")
print("   Preheat oven to 350°F, mix ingredients, and bake for 30 minutes.'")
print()
print("Response B (REJECTED):")
print("  'Cake is a type of food. It can be made in many ways. Food is important.")
print("   People eat food every day. Cake is popular at birthday parties.'")
print()
print("The reward model learns to give A a higher score than B.")
print("This signal is then used to update the LLM to prefer responses like A.")

In [ ]:
# Implement DPO loss from scratch

def dpo_loss(model_logprobs_chosen, model_logprobs_rejected,
             ref_logprobs_chosen, ref_logprobs_rejected, beta=0.1):
    """
    Direct Preference Optimization loss.
    
    Encourages the model to increase probability of chosen responses
    and decrease probability of rejected responses, relative to reference.
    """
    # Log-ratio of policy vs reference for chosen/rejected
    chosen_ratio = model_logprobs_chosen - ref_logprobs_chosen
    rejected_ratio = model_logprobs_rejected - ref_logprobs_rejected
    
    # DPO loss: maximize the gap between chosen and rejected
    loss = -F.logsigmoid(beta * (chosen_ratio - rejected_ratio)).mean()
    
    # Metrics for monitoring
    chosen_reward = beta * chosen_ratio.detach().mean()
    rejected_reward = beta * rejected_ratio.detach().mean()
    reward_margin = (chosen_reward - rejected_reward).item()
    
    return loss, {
        'loss': loss.item(),
        'chosen_reward': chosen_reward.item(),
        'rejected_reward': rejected_reward.item(),
        'reward_margin': reward_margin,
    }

# Demonstrate with dummy data
batch_size = 4
chosen_logp = torch.randn(batch_size) - 1  # Model log-probs for chosen
rejected_logp = torch.randn(batch_size) - 2  # Model log-probs for rejected
ref_chosen = torch.randn(batch_size) - 1  # Reference log-probs
ref_rejected = torch.randn(batch_size) - 2

loss, metrics = dpo_loss(chosen_logp, rejected_logp, ref_chosen, ref_rejected)
print(f'DPO Loss: {metrics["loss"]:.4f}')
print(f'Chosen reward:  {metrics["chosen_reward"]:.4f}')
print(f'Rejected reward: {metrics["rejected_reward"]:.4f}')
print(f'Reward margin:   {metrics["reward_margin"]:.4f} (should increase during training)')

## 6. Exercises

### Exercise 1: Fine-tune on Custom Data
Create your own instruction dataset (at least 20 examples) on a topic you care about. Fine-tune GPT-2 using LoRA and evaluate the results.

In [ ]:
# EXERCISE 1: Custom fine-tuning
# Your code here

### Exercise 2: LoRA Rank Ablation
Fine-tune the same model with different LoRA ranks (2, 4, 8, 16, 32, 64). Plot:
- Final loss vs rank
- Number of trainable parameters vs rank
- Generation quality (manual evaluation) vs rank

What's the sweet spot?

In [ ]:
# EXERCISE 2: LoRA rank ablation
# Your code here

### Exercise 3: Implement DPO Training
Create a preference dataset with chosen/rejected response pairs. Implement a full DPO training loop and show the reward margin increasing over training.

In [ ]:
# EXERCISE 3: DPO training
# Your code here

## 7. Key Takeaways

1. **Pre-training** uses next-token prediction on massive text corpora to learn language
2. **Key training tricks**: LR warmup + cosine decay, gradient accumulation, mixed precision
3. **SFT** teaches the model to follow instructions using curated prompt-response pairs
4. **Loss masking** ensures the model only learns from response tokens, not prompts
5. **LoRA** makes fine-tuning practical by only training ~1% of parameters
6. **RLHF/DPO** aligns the model with human preferences for safety and helpfulness

### The Full Pipeline
```
Raw Text Data → Pre-training → Base Model → SFT → Instruction Model → RLHF/DPO → Chatbot
```

### Next Up
**Notebook 5: Building a Complete Chatbot** — We'll use everything we've learned to build, deploy, and evaluate a functional chatbot with conversation memory, RAG, and tool use.